# Variant-to-function evidence agent: Colab runner
Run the cells **top to bottom, one at a time**. No zip needed: it finds the loose AlphaGenome 2 parquet files in your Google Drive.
Not yet verified against live services: the Ensembl lookup (section 3) and Open Targets (section 7). If either fails, the cell prints what to paste back.

## 1. Setup (installs everything; safe to re-run)

In [ ]:
import os, sys, subprocess, glob
from google.colab import drive
drive.mount('/content/drive')

def sh(*cmd):
    r = subprocess.run(list(cmd), capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode != 0:
        raise RuntimeError(f"command failed: {' '.join(cmd)}\n{out[-1500:]}")
    return out

REPO_URL = 'https://github.com/Raj4884/Variant-to-function-evidence-agent.git'
if os.path.isdir('/content/repo/.git'):
    sh('git', '-C', '/content/repo', 'pull', '-q')
else:
    sh('git', 'clone', '-q', REPO_URL, '/content/repo')
sh(sys.executable, '-m', 'pip', 'install', '-q', 'duckdb', 'pandas', 'pyarrow', 'pydantic', 'httpx', 'pyyaml', 'pytest', 'anthropic')
if '/content/repo/src' not in sys.path:
    sys.path.insert(0, '/content/repo/src')   # no editable install needed
for m in [m for m in sys.modules if m.startswith('variant_evidence')]:
    del sys.modules[m]   # drop stale copies so the freshly pulled code is what runs
import variant_evidence.resolve, variant_evidence.convert, variant_evidence.agent
env = dict(os.environ, PYTHONPATH='/content/repo/src')
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '/content/repo/tests'], capture_output=True, text=True, env=env, cwd='/content/repo')
print(r.stdout.strip().splitlines()[-1])

## 2. Find your AlphaGenome 2 files in Drive and inspect them

In [ ]:
import duckdb
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f'{name} not found anywhere in My Drive. Check it is uploaded to this Google account.')
    return hits[0]
EXPR = find('variant_scores_sc_gene_expression.parquet')
VARIANTS = find('variants.parquet')
print(EXPR)
con = duckdb.connect()
print(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{EXPR}')").fetchdf()[['column_name','column_type']].to_string())
print(con.execute(f"SELECT * FROM read_parquet('{EXPR}') LIMIT 2").fetchdf().T.to_string())
print(con.execute(f"SELECT channel_name, count(*) AS n FROM read_parquet('{EXPR}') GROUP BY 1 ORDER BY n DESC").fetchdf().to_string())

## 3. Turn rsIDs into genome coordinates
The AG2 files contain no rsIDs, only hg38 coordinates, so the rsIDs are looked up through the Ensembl API. If this cell fails, paste the error.

In [ ]:
from variant_evidence.resolve import resolve_rsids
RSIDS = ['rs2836882','rs12913832','rs6497271','rs1175550','rs6673829','rs2038740',
         'rs13066062','rs35705950','rs3820282','rs1421085','rs12740374']
PRIMARY = {str(i) for i in range(1, 23)} | {'X', 'Y'}
coords = resolve_rsids(RSIDS)
cand = {c: r for r, cs in coords.items() for c in cs if c[0] in PRIMARY}   # drop alt-contig duplicates

# keep only alleles that exist in the AG2 variant panel (so alt alleles are never mixed up)
pl = ','.join(str(p) for p in sorted({c[1] for c in cand}))
panel = con.execute(f"SELECT chromosome, position, ref, alt FROM read_parquet('{VARIANTS}') WHERE position IN ({pl})").fetchall()
in_panel = {(str(c).replace('chr', ''), int(p), r, a) for c, p, r, a in panel}
coord_to_rsid = {c: r for c, r in cand.items() if c in in_panel}
scored_rsids = set(coord_to_rsid.values())
for r in RSIDS:
    alleles = [c for c, rr in coord_to_rsid.items() if rr == r]
    print(r, alleles if alleles else 'NOT IN AG2 PANEL')
    if len(alleles) > 1: print('   WARNING: more than one alt allele in panel; check before trusting direction')

## 4. Pull just those variants out of the big file and normalise
Reads the 1.1 GB file with DuckDB (a few minutes). The printed effect summary is how we check the units: the paper's call threshold is |LFC| > 0.1.

In [ ]:
import pandas as pd
from variant_evidence.convert import normalise_ag2_expression, collapse_channels, write_normalised_csv
LFC_SCALE = 1.0      # set to math.log(2) only if the units turn out to be log2
OUT = '/content/drive/MyDrive/ag2_normalised.csv'

pos_list = ','.join(str(p) for p in sorted({c[1] for c in coord_to_rsid}))
cur = con.execute(f"SELECT * FROM read_parquet('{EXPR}') WHERE position IN ({pos_list})")
cols = [d[0] for d in cur.description]
per_channel = normalise_ag2_expression((dict(zip(cols, r)) for r in cur.fetchall()), coord_to_rsid, lfc_scale=LFC_SCALE)
n = write_normalised_csv(collapse_channels(per_channel), OUT)   # one row per cell type, median across assay channels
print(n, 'rows written to', OUT)
df = pd.read_csv(OUT)
print(df.groupby('variant').size().to_string())
print(df.lfc.describe().to_string())
print(df.n_agree.value_counts().sort_index().to_string())   # how many assay channels agree per row

## 5. Run the agent on every variant

In [ ]:
from variant_evidence.agent import run_agent
from variant_evidence.models import VariantQuery
from variant_evidence.providers.ag2_precomputed import AG2PrecomputedProvider

providers = [AG2PrecomputedProvider(OUT, scored_rsids=scored_rsids)]
reports = {}
for rsid in RSIDS:
    rep = run_agent(VariantQuery(rsid=rsid), providers)
    reports[rsid] = rep
    if not rep.hypotheses:
        print(f'{rsid}: ABSTAIN (no called gene effect)'); continue
    h = rep.hypotheses[0]
    print(f'{rsid}: {h.gene} {h.direction.value} conf={h.confidence.value} cells={h.top_cell_types[:2]}')
    for note in h.notes: print('    NOTE:', note)
print()
for rsid in ['rs1421085']:
    print(rsid, 'caveats:', reports[rsid].caveats)
print()
print('rs12740374 restricted to liver (the tissue implied by its trait, LDL cholesterol):')
rep = run_agent(VariantQuery(rsid='rs12740374'), providers, context=['liver'])
for h in rep.hypotheses[:4]:
    print(f'  {h.gene:8s} {h.direction.value:5s} score={h.score:.2f} conf={h.confidence.value}')
print(rep.hypotheses[0].notes if rep.hypotheses else rep.caveats)

## 6. Evaluation on the gold loci

In [ ]:
r = subprocess.run([sys.executable, '/content/repo/eval/run_eval.py', '--ag2', OUT], capture_output=True, text=True, env=env, cwd='/content/repo')
print(r.stdout + r.stderr)

## 7. Open Targets: fine-mapping and gene predictions (live)
For each variant: its fine-mapping probability (PIP) and Open Targets' own best gene (L2G), next to the AG2 result and the gold gene.
L2G is the baseline the agent has to beat. Everything is also saved to `ot_result.txt` in your Drive. If a variant errors, the message says why.

In [ ]:
import yaml, warnings
from variant_evidence.providers import open_targets as ot
from variant_evidence.baselines import l2g_top_gene
from variant_evidence.aggregator import best_pip

gold = {g['rsid']: g for g in yaml.safe_load(open('/content/repo/eval/gold.yaml'))['loci']}
ot_provider = ot.OpenTargetsProvider(ot.http_transport())
lines = ['rsid         gold    AG2(alone)  OT-L2G(score)     best_PIP  n_cred_sets']
ok = []
for rsid in RSIDS:
    try:
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter('always')
            items = ot_provider.fetch(VariantQuery(rsid=rsid))
        pip, _ = best_pip(items)
        l2g = l2g_top_gene(items)
        ag2 = reports[rsid].hypotheses[0].gene if reports[rsid].hypotheses else 'abstain'
        l2g_s = f"{l2g[0]} ({l2g[1]:.2f})" if l2g else '-'
        n_cs = sum(1 for i in items if i.kind == 'credible_set')
        lines.append(f"{rsid:12s} {gold[rsid]['gene']:7s} {ag2:10s}  {l2g_s:16s}  {pip if pip is None else round(pip, 3)!s:8s}  {n_cs}" + (f"   WARN {w[0].message}" if w else ''))
        ok.append(rsid)
    except Exception as e:
        lines.append(f"{rsid:12s} ERROR {str(e)[:300]}")
print('\n'.join(lines))

# agent with both evidence sources
lines2 = ['', 'AG2 + Open Targets together (no tissue context):']
both = [AG2PrecomputedProvider(OUT, scored_rsids=scored_rsids), ot_provider]
for rsid in ok:
    try:
        rep = run_agent(VariantQuery(rsid=rsid), both)
        if not rep.hypotheses:
            lines2.append(f'{rsid}: ABSTAIN'); continue
        h = rep.hypotheses[0]
        lines2.append(f'{rsid}: {h.gene} {h.direction.value} conf={h.confidence.value}')
        lines2 += ['    NOTE: ' + x for x in h.notes]
    except Exception as e:
        lines2.append(f'{rsid}: ERROR {str(e)[:200]}')
print('\n'.join(lines2))
open('/content/drive/MyDrive/ot_result.txt', 'w').write('\n'.join(lines + lines2))

## 8. Held-out evaluation on literature-validated variant-gene pairs
Variants come from the Alsheikh 2022 review table (selection rules are written in `src/variant_evidence/validated.py` and were fixed before any results were seen).
Needs `validated_sheet0.csv` in your Drive (saved by the earlier download cell). Only the gene is scored, tissue-agnostic. Takes a few minutes.

In [ ]:
import pandas as pd, yaml
from variant_evidence.validated import select_validated

tbl = pd.read_csv(find('validated_sheet0.csv'))
excl = {g['rsid'] for g in yaml.safe_load(open('/content/repo/eval/gold.yaml'))['loci']}
sel = select_validated(tbl, excl)
sel['pos_hg38'] = sel['pos_hg38'].astype(int)
print(len(sel), 'variants pass the selection rules before the panel check')

# R6: single-alt SNV in the AG2 variant panel at the listed hg38 position
con.register('sel', sel[['rsid', 'chr', 'pos_hg38']])
pan = con.execute(f"""SELECT s.rsid, p.chromosome, p.position, p.ref, p.alt
    FROM sel s JOIN read_parquet('{VARIANTS}') p ON p.position = s.pos_hg38 AND p.chromosome = s.chr
    WHERE length(p.ref) = 1 AND length(p.alt) = 1""").fetchdf()
one_alt = pan.groupby('rsid').size()
pan = pan[pan.rsid.isin(one_alt[one_alt == 1].index)]
vcoord = {(str(c).replace('chr', ''), int(p), r, a): rs for rs, c, p, r, a in pan.itertuples(index=False)}
sel = sel[sel.rsid.isin(set(vcoord.values()))].reset_index(drop=True)
print(len(sel), 'usable: single-alt SNV present in the AG2 panel')
print(sel.stratum.value_counts().to_string())

In [ ]:
from variant_evidence.convert import normalise_ag2_expression, collapse_channels, write_normalised_csv
OUT2 = '/content/drive/MyDrive/ag2_validated.csv'
pl = ','.join(str(p) for p in sorted({c[1] for c in vcoord}))
cur = con.execute(f"SELECT * FROM read_parquet('{EXPR}') WHERE position IN ({pl})")
cols = [d[0] for d in cur.description]
n = write_normalised_csv(collapse_channels(normalise_ag2_expression((dict(zip(cols, r)) for r in cur.fetchall()), vcoord)), OUT2)
print(n, 'rows written')

prov = AG2PrecomputedProvider(OUT2, scored_rsids=set(vcoord.values()))
res = []
for row in sel.itertuples():
    rep = run_agent(VariantQuery(rsid=row.rsid), [prov])
    top = rep.hypotheses[0] if rep.hypotheses else None
    res.append(dict(rsid=row.rsid, stratum=row.stratum, genes='/'.join(row.genes), trait=row.trait,
                    ag2_gene=top.gene if top else 'abstain', ag2_conf=top.confidence.value if top else '-',
                    ag2_hit=bool(top and top.gene in row.genes), ag2_abstain=top is None))
res = pd.DataFrame(res)

def summarise(res, cols):
    out = []
    for st, g in [('ALL', res)] + list(res.groupby('stratum')):
        line = f"{st:9s} n={len(g):3d}"
        for c in cols:
            line += f"  {c}: hit {int(g[c + '_hit'].sum())}/{len(g)}"
        out.append(line)
    return out
lines = summarise(res, ['ag2'])
print('\n'.join(lines))
print('abstained:', int(res.ag2_abstain.sum()), '| hits among non-abstained:', int(res.ag2_hit.sum()), '/', int((~res.ag2_abstain).sum()))
res.to_csv('/content/drive/MyDrive/validated_results.csv', index=False)
open('/content/drive/MyDrive/validated_results.txt', 'w').write('\n'.join(lines))

### 8b. Open Targets L2G on the same variants (slow: 10+ minutes)
Saves progress to Drive as it goes. If it is interrupted, re-run it: finished variants are skipped.

In [ ]:
import time, os
from variant_evidence.providers import open_targets as ot
from variant_evidence.baselines import l2g_top_gene
P = '/content/drive/MyDrive/validated_l2g.csv'
done = pd.read_csv(P) if os.path.exists(P) else pd.DataFrame(columns=['rsid', 'l2g_gene', 'l2g_score', 'err'])
seen = set(done.rsid)
prov_ot = ot.OpenTargetsProvider(ot.http_transport())
t0 = time.time()
rows = done.to_dict('records')
for rsid in res.rsid:
    if rsid in seen: continue
    try:
        l2g = l2g_top_gene(prov_ot.fetch(VariantQuery(rsid=rsid)))
        rows.append(dict(rsid=rsid, l2g_gene=l2g[0] if l2g else '-', l2g_score=l2g[1] if l2g else None, err=''))
    except Exception as e:
        rows.append(dict(rsid=rsid, l2g_gene='-', l2g_score=None, err=str(e)[:120]))
    pd.DataFrame(rows).to_csv(P, index=False)
    if time.time() - t0 > 1500: print('time budget reached; re-run this cell to continue'); break
l2 = pd.DataFrame(rows)
m = res.merge(l2, on='rsid', how='left')
m['l2g_hit'] = [(g in genes.split('/')) for g, genes in zip(m.l2g_gene.fillna('-'), m.genes)]
m['either_hit'] = m.ag2_hit | m.l2g_hit
print('L2G finished for', len(l2), 'of', len(res), '| errors:', int((l2.err != '').sum()))
lines = summarise(m[m.rsid.isin(l2.rsid)], ['ag2', 'l2g', 'either'])
print('\n'.join(lines))
m.to_csv('/content/drive/MyDrive/validated_results_with_l2g.csv', index=False)
open('/content/drive/MyDrive/validated_results.txt', 'a').write('\n\nWith L2G:\n' + '\n'.join(lines))

## 9. What to paste back into the chat
Tell the chat when section 7 is done. The results are saved to `ot_result.txt` in your Drive and can be read from there. Also paste any error text.